In [5]:
import os
from sedona.spark import SedonaContext
import geopandas as gpd
from time import time
from sedona.spark import dataframe_to_arrow
import geopandas as gpd
from sedona.spark.geoarrow import create_spatial_dataframe
from sedona.spark.maps.SedonaKepler import SedonaKepler

In [6]:
import pyspark.sql.functions as f
import matplotlib.pyplot as plt
import geopandas as gpd

In [7]:
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

In [8]:
def create_sedona_session(config_params):
    config = SedonaContext.builder()

    for key, value in config_params.items():
        config = config.config(key, value)


    sedona = SedonaContext.create(config.getOrCreate())
    sedona.sparkContext.setLogLevel("ERROR")

    return sedona
    
    
class SedonaBenchmark():
    
    def __init__(self, config: dict):
        self.sedona = create_sedona_session(config)
        
    def __enter__(self):
        return self.sedona
        
    def __exit__(self, type, value, traceback):
        self.sedona.stop()

In [9]:
from dataclasses import dataclass
import time

@dataclass
class GeometryGenerationParams:
    size: int
    geometry_type: str

    def get_scale(self, other):
        return max(self.size, other.size)

    
def create_geometries(param: GeometryGenerationParams, other_param):
    partitions = int(param.size/150_000) + 1
    scale = param.get_scale(other_param)

    df = sedona.read.format("spider").load(
        n=param.size,
        geometryType=param.geometry_type,
        maxSize=0.01,
        seed=43,
        scaleX=scale,
        scaleY=scale,
        numPartitions=partitions,
    )

    return df


def time_it_with_container(l, fn):
    start = time.time()
    fn()
    took = time.time() - start
    l.append(took)
    print(f"process took {took}")
        
def spatial_join(left, right, explain=False):
    res = left.alias("l").\
        join(right.alias("r"), f.expr("ST_Intersects(l.geometry, r.geometry)")).\
        select("l.id", "l.geometry")

    if explain:
        res.explain()

    res.write.mode("overwrite").format("noop").save()

    if explain:
        from sedona.utils.adapter import Adapter
        srdd = Adapter.toSpatialRdd(res, "geometry")
        print(srdd.getPartitioner())
    

In [10]:
with SedonaBenchmark({}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=10_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x==1))

    print(sum(container)/len(container))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/25 20:32:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/08/25 20:32:35 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/08/25 20:32:35 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/08/25 20:32:35 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/08/25 20:32:35 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/08/25 20:32:35 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/08/25 20:32:35 WARN SimpleFunctionRegistry: The function st_envelop

process took 0.5650429725646973
== Physical Plan ==
*(3) Project [id#44L, geometry#45]
+- RangeJoin geometry#45: geometry, geometry#49: geometry, INTERSECTS
   :- *(1) Project [id#44L, geometry#45]
   :  +- *(1) Filter isnotnull(geometry#45)
   :     +- BatchScan spider[id#44L, geometry#45] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []
   +- *(2) Project [geometry#49]
      +- *(2) Filter isnotnull(geometry#49)
         +- BatchScan spider[id#48L, geometry#49] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []




/tmp/ipykernel_321/191528201.py:48: DeprecationWarning: Importing from 'sedona.utils.adapter' is deprecated. Please use 'sedona.spark.utils.adapter' instead.
  from sedona.utils.adapter import Adapter


SpatialPartitioner(name=None, jvm_partitioner=None)
process took 0.27675843238830566
process took 0.13035082817077637
process took 0.13915657997131348
process took 0.12184953689575195
process took 0.13996624946594238
process took 0.279482364654541
process took 0.15705418586730957
process took 0.09249496459960938
process took 0.11765480041503906
0.2019810914993286


+-------+
|time[s]|
+-------+
|   0.57|
|   0.28|
|   0.13|
|   0.14|
|   0.12|
|   0.14|
|   0.28|
|   0.16|
|   0.09|
|   0.12|
+-------+



In [84]:
# removing optimizations

In [11]:
with SedonaBenchmark({"sedona.join.optimizationmode": "none"}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=10_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points))
    print(sum(container)/len(container))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

process took 15.93944001197815


process took 15.6744225025177


process took 16.063696146011353


process took 19.856091260910034


process took 21.454818725585938


process took 17.065799951553345


process took 15.992290496826172


process took 16.367636919021606


process took 17.07321047782898


process took 16.626783847808838
17.211419034004212
+-------+
|time[s]|
+-------+
|  15.94|
|  15.67|
|  16.06|
|  19.86|
|  21.45|
|  17.07|
|  15.99|
|  16.37|
|  17.07|
|  16.63|
+-------+



# broadcast vs non broadcast join

In [ ]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({"sedona.join.autoBroadcastJoinThreshold": "100MB", "spark.sql.autoBroadcastJoinThreshold": "50MB"}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=10_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=100_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        # uncomment it, if you want to have SQL version of it
        # polygons.createOrReplaceTempView("left")
        # points.createOrReplaceTempView("right")

        # sedona.sql(
        #     """
        #     SELECT
        #         /*+ BROADCASTJOIN (l) */
        #         l.id,
        #         l.geometry
        #     FROM right AS r
        #     JOIN left AS l ON ST_Intersects(r.geometry, l.geometry) 
        #     """
        # ).explain()

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, broadcast(points), explain=x == 0))
    print(sum(container)/len(container))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

In [ ]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({
    "sedona.join.autoBroadcastJoinThreshold": "-1",
    "spark.driver.memory": "12G",
    "spark.executor.memory": "16G",
}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=10_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=100_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    print(sum(container)/len(container))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

# Global index turn on and turn off for smaller datasets

In [18]:
from sedona.spark.core.SpatialRDD.spatial_rdd import SpatialRDD

In [19]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({"sedona.global.index": "False"}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=1000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=1000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    print(sum(container)/len(container))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

== Physical Plan ==
*(3) Project [id#918L, geometry#919]
+- RangeJoin geometry#919: geometry, geometry#923: geometry, INTERSECTS
   :- *(1) Project [id#918L, geometry#919]
   :  +- *(1) Filter isnotnull(geometry#919)
   :     +- BatchScan spider[id#918L, geometry#919] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []
   +- *(2) Project [geometry#923]
      +- *(2) Filter isnotnull(geometry#923)
         +- BatchScan spider[id#922L, geometry#923] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []


SpatialPartitioner(name=None, jvm_partitioner=None)
process took 0.10137486457824707
process took 0.056601524353027344
process took 0.05685710906982422
process took 0.04949140548706055
process took 0.06927227973937988
process took 0.051070451736450195
process took 0.05171608924865723
process took 0.05455160140991211
process took 0.05194091796875
process took 0.04590344429016113
0.058877968788146974
+-------+
|t

In [20]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=1000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=1000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    print(sum(container)/len(container))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

== Physical Plan ==
*(3) Project [id#1144L, geometry#1145]
+- RangeJoin geometry#1145: geometry, geometry#1149: geometry, INTERSECTS
   :- *(1) Project [id#1144L, geometry#1145]
   :  +- *(1) Filter isnotnull(geometry#1145)
   :     +- BatchScan spider[id#1144L, geometry#1145] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []
   +- *(2) Project [geometry#1149]
      +- *(2) Filter isnotnull(geometry#1149)
         +- BatchScan spider[id#1148L, geometry#1149] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []


SpatialPartitioner(name=None, jvm_partitioner=None)
process took 0.13041019439697266
process took 0.0515749454498291
process took 0.05572843551635742
process took 0.04452705383300781
process took 0.03579974174499512
process took 0.03989362716674805
process took 0.07413268089294434
process took 0.03582596778869629
process took 0.033417701721191406
process took 0.04128146171569824
0.05425918102264404

# Global index turn on and turn off for larger datasets

In [ ]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({"sedona.global.index": "False"}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=100_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

In [ ]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({"sedona.global.index": "True"}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=100_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    print(sum(container)/len(container))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

# Changing Number of partitions

In [13]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({"sedona.join.numpartition": 2, "app-name": "sedona"}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=100_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

== Physical Plan ==
*(3) Project [id#1474L, geometry#1475]
+- RangeJoin geometry#1475: geometry, geometry#1479: geometry, INTERSECTS
   :- *(1) Project [id#1474L, geometry#1475]
   :  +- *(1) Filter isnotnull(geometry#1475)
   :     +- BatchScan spider[id#1474L, geometry#1475] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []
   +- *(2) Project [geometry#1479]
      +- *(2) Filter isnotnull(geometry#1479)
         +- BatchScan spider[id#1478L, geometry#1479] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []




process took 2.700004816055298


process took 3.443556547164917


process took 2.4503591060638428


process took 2.534167766571045


...

process took 2.249656915664673


process took 2.199169397354126


process took 2.3127999305725098


process took 2.221738815307617


process took 2.6041383743286133
+-------+
|time[s]|
+-------+
|    2.7|
|   3.44|
|   2.45|
|   2.53|
|    2.8|
|   2.25|
|    2.2|
|   2.31|
|   2.22|
|    2.6|
+-------+



In [40]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({"sedona.join.numpartition": 200}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

== Physical Plan ==
*(3) Project [id#6498L, geometry#6499]
+- RangeJoin geometry#6499: geometry, geometry#6503: geometry, INTERSECTS
   :- *(1) Project [id#6498L, geometry#6499]
   :  +- *(1) Filter isnotnull(geometry#6499)
   :     +- BatchScan spider[id#6498L, geometry#6499] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []
   +- *(2) Project [geometry#6503]
      +- *(2) Filter isnotnull(geometry#6503)
         +- BatchScan spider[id#6502L, geometry#6503] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []




process took 43.459760904312134


process took 41.03830027580261


process took 41.573967695236206


process took 45.844337940216064


...

process took 42.533074378967285


process took 41.51686930656433


process took 41.45458507537842


process took 43.23405194282532


process took 41.74102544784546
+-------+
|time[s]|
+-------+
|  43.46|
|  41.04|
|  41.57|
|  45.84|
|  41.88|
|  42.53|
|  41.52|
|  41.45|
|  43.23|
|  41.74|
+-------+



In [ ]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({"sedona.join.numpartition": 5}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

In [6]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({"sedona.join.numpartition": 100}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

== Physical Plan ==
*(3) Project [id#46L, geometry#47]
+- RangeJoin geometry#47: geometry, geometry#51: geometry, INTERSECTS
   :- *(1) Project [id#46L, geometry#47]
   :  +- *(1) Filter isnotnull(geometry#47)
   :     +- BatchScan spider[id#46L, geometry#47] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []
   +- *(2) Project [geometry#51]
      +- *(2) Filter isnotnull(geometry#51)
         +- BatchScan spider[id#50L, geometry#51] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []




process took 50.050769329071045


process took 46.52316403388977


process took 49.60397529602051


process took 45.30474853515625


...

process took 50.381609201431274


process took 45.69617795944214


process took 45.95240783691406


process took 48.470231771469116


process took 46.62135934829712
+-------+
|time[s]|
+-------+
|  50.05|
|  46.52|
|   49.6|
|   45.3|
|  52.44|
|  50.38|
|   45.7|
|  45.95|
|  48.47|
|  46.62|
+-------+



In [7]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({"sedona.join.numpartition": 60}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

== Physical Plan ==
*(3) Project [id#271L, geometry#272]
+- RangeJoin geometry#272: geometry, geometry#276: geometry, INTERSECTS
   :- *(1) Project [id#271L, geometry#272]
   :  +- *(1) Filter isnotnull(geometry#272)
   :     +- BatchScan spider[id#271L, geometry#272] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []
   +- *(2) Project [geometry#276]
      +- *(2) Filter isnotnull(geometry#276)
         +- BatchScan spider[id#275L, geometry#276] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []




process took 48.11615872383118


process took 46.55289077758789


process took 47.75520157814026


process took 47.906208753585815


...

process took 51.064319133758545


process took 50.2182354927063


process took 50.62295436859131


process took 48.15794515609741


process took 48.79333829879761
+-------+
|time[s]|
+-------+
|  48.12|
|  46.55|
|  47.76|
|  47.91|
|  49.81|
|  51.06|
|  50.22|
|  50.62|
|  48.16|
|  48.79|
+-------+



In [ ]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({"sedona.join.numpartition": 35, "spark.executor.instances": 3, "spark.executor.cores": 3}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

In [ ]:
import time
with SedonaBenchmark({"sedona.join.numpartition": 250}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

In [10]:
import time
with SedonaBenchmark({"sedona.join.numpartition": 1500}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

== Physical Plan ==
*(3) Project [id#946L, geometry#947]
+- RangeJoin geometry#947: geometry, geometry#951: geometry, INTERSECTS
   :- *(1) Project [id#946L, geometry#947]
   :  +- *(1) Filter isnotnull(geometry#947)
   :     +- BatchScan spider[id#946L, geometry#947] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []
   +- *(2) Project [geometry#951]
      +- *(2) Filter isnotnull(geometry#951)
         +- BatchScan spider[id#950L, geometry#951] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []




process took 42.06498169898987


process took 40.69003891944885


process took 39.95975613594055


process took 41.96866464614868


...

process took 39.82334852218628


process took 42.52166724205017


process took 41.12245488166809


process took 40.83388662338257


process took 40.29218149185181
+-------+
|time[s]|
+-------+
|  42.06|
|  40.69|
|  39.96|
|  41.97|
|   40.5|
|  39.82|
|  42.52|
|  41.12|
|  40.83|
|  40.29|
+-------+



In [ ]:
import time
with SedonaBenchmark({"sedona.join.numpartition": 10000}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

# changing the partitioning

In [ ]:
import time

with SedonaBenchmark({"sedona.join.gridtype": "quadtree"}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=100_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

In [26]:
import time
with SedonaBenchmark({"sedona.join.gridtype": "kdbtree"}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=100_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

== Physical Plan ==
*(3) Project [id#3332L, geometry#3333]
+- RangeJoin geometry#3333: geometry, geometry#3337: geometry, INTERSECTS
   :- *(1) Project [id#3332L, geometry#3333]
   :  +- *(1) Filter isnotnull(geometry#3333)
   :     +- BatchScan spider[id#3332L, geometry#3333] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []
   +- *(2) Project [geometry#3337]
      +- *(2) Filter isnotnull(geometry#3337)
         +- BatchScan spider[id#3336L, geometry#3337] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []




process took 11.179224014282227


process took 11.652714729309082


process took 11.694740056991577


process took 12.64444613456726


...

process took 17.623416662216187


process took 15.683395385742188


process took 18.793715000152588


process took 20.21807050704956


process took 11.519941568374634
+-------+
|time[s]|
+-------+
|  11.18|
|  11.65|
|  11.69|
|  12.64|
|  12.01|
|  17.62|
|  15.68|
|  18.79|
|  20.22|
|  11.52|
+-------+

